# Interactive 3D Visualization of Optimal Allosteric Paths

This notebook provides an interactive way to visualize the optimal allosteric paths calculated for Wild-Type (WT) and Mutant (Y138H) protein structures. 

You can use the dropdown menus below to select:
1.  **System**: Choose between the WT and Mutant protein.
2.  **Category**: Select a specific category of allosteric paths to display. The `combined high contact` category aggregates all paths from the three `high contact` categories.

In [1]:
import re
import os
from collections import defaultdict
import ipywidgets as widgets
from ipywidgets import interact, Dropdown
import py3Dmol

## 1. Parse Optimal Path Data

First, we parse the `optimal_paths_details.md` file. The function below reads the markdown file, identifies the different categories of paths, and extracts the residue lists for each path under the corresponding system (WT or Mutant).

In [2]:
def parse_optimal_paths_by_category(markdown_file):
    """
    Parses the optimal path data from a markdown file, grouping paths by category and system.
    
    Args:
        markdown_file (str): The path to the markdown file.
        
    Returns:
        defaultdict: A nested dictionary with the structure {category: {system: [path_residues]}}.
    """
    with open(markdown_file, 'r') as f:
        content = f.read()

    # Split the content by '## Category: ' headers to process each category section separately
    sections = re.split(r'## Category: ', content)
    all_data = defaultdict(lambda: defaultdict(list))

    for section in sections:
        if not section.strip():
            continue
        
        lines = section.strip().split('\n')
        category_name = lines[0].strip()
        
        for line in lines:
            # Identify table rows, which start with '|'
            if not line.strip().startswith('|'):
                continue
            
            cols = [c.strip() for c in line.split('|')]
            
            # A valid data row has at least 8 columns (including empty ones from split)
            # and the 'System' column (cols[1]) is either 'WT' or 'Mutant'
            if len(cols) > 7 and cols[1] in ['WT', 'Mutant']:
                system = cols[1]
                path_str = cols[7]
                
                if "N/A" not in path_str:
                    path_residues = [int(r) for r in re.findall(r'\d+', path_str)]
                    if path_residues:
                        all_data[category_name][system].append(path_residues)
                        
    return all_data

# Load the data
markdown_path = '../analysis_results/optimal_paths_details.md'
all_path_data = parse_optimal_paths_by_category(markdown_path)

# Create the special 'combined high contact' category
combined_high_contact = defaultdict(list)
for category in ['high contact both', 'high contact WT', 'high contact Mutant']:
    if category in all_path_data:
        for system in ['WT', 'Mutant']:
            if system in all_path_data[category]:
                combined_high_contact[system].extend(all_path_data[category][system])
all_path_data['combined high contact'] = combined_high_contact

print(f"Loaded {len(all_path_data)} categories: {list(all_path_data.keys())}")

Loaded 5 categories: ['gamma loop', 'high contact Mutant', 'high contact WT', 'high contact both', 'combined high contact']


## 2. Interactive 3D Visualization

The following cell sets up the interactive `py3Dmol` viewer. When you select a system and category from the dropdowns, the `update_view` function is called to render the corresponding PDB structure and overlay the selected paths.

- The protein is shown as a grey cartoon.
- The C-alpha atoms of residues in a path are shown as colored spheres.
- The path itself is traced with cylinders connecting the C-alpha atoms.

In [3]:
# Define paths to PDB files
pdb_files = {
    'WT': '../Data/AF2_LM211_WT/calcium/frame1.pdb',
    'Mutant': '../Data/AF2_LM2_Y138H_11_Mutant/calcium/frame1.pdb'
}

# Function to be called by the interactive widgets
def update_view(system, category):
    pdb_path = pdb_files[system]
    
    # Create a py3Dmol view
    view = py3Dmol.view(width=800, height=600)
    
    # Add the protein model
    view.addModel(open(pdb_path, 'r').read(), 'pdb')
    
    # Set a basic cartoon style
    view.setStyle({'cartoon': {'color': 'grey'}})
    
    # Get the selected paths
    paths = all_path_data.get(category, {}).get(system, [])
    
    # Define a color palette for the paths
    colors = ['#FF0000', '#00FF00', '#0000FF', '#FFFF00', '#FF00FF', '#00FFFF', '#FFA500', '#800080', '#FFC0CB', '#A52A2A']
    
    if not paths:
        print(f"No paths found for System: {system}, Category: {category}")
    
    # Draw each path
    for i, path in enumerate(paths):
        color = colors[i % len(colors)]
        
        # Style the residues in the path as spheres
        view.addStyle({'resi': path, 'atom': 'CA'}, {'sphere': {'color': color, 'radius': 0.6}})
        
        # Add cylinders between C-alpha atoms to visualize the path flow
        for j in range(len(path) - 1):
            res1 = path[j]
            res2 = path[j+1]
            view.addCylinder({
                'start': {'resi': res1, 'atom': 'CA'},
                'end': {'resi': res2, 'atom': 'CA'},
                'color': color,
                'radius': 0.2
            })
            
    view.zoomTo()
    view.show()

# Create dropdown widgets
system_widget = Dropdown(options=['WT', 'Mutant'], description='System:')
category_widget = Dropdown(options=sorted(list(all_path_data.keys())), description='Category:')

# Use interact to link the widgets to the update function
interact(update_view, system=system_widget, category=category_widget);

interactive(children=(Dropdown(description='System:', options=('WT', 'Mutant'), value='WT'), Dropdown(descript…